# Cleaning the enriched dataset and selecting the corpus

The following python code walks through the proces of creating a balanced dataset for stylometric anlysis. The names of the authors have already been standardized in OpenRefine, and metadata such a the publication date of the novel and the gender and country of citizenship have been reconciled using OpenRefine. The Downloads column represents the number of downloads from the Project Gutenberg webiste, and will be the basis for the selection of novels, alongside country of citizenship.

The goal is to standardize genre and country of citizenship, and to make a balanced selection of novels

In [ ]:
import pandas as pd

df = pd.read_csv("sf-fantasy-novels-csv-enriched.csv")

df.head(10)

### Filtering the data on UK and US authorship

The United Kingdom has had multiple names over history:
- United Kingdom
- United Kingdom of Great Britain and Ireland
- United Kingdom of Great Britain
- British Empire

The reconciled values from wikidata have to be standardized into "United Kingdom" for the purpose of later analysis. The following function standardizes these values. This is not neccesary for the United States: these values are already consistant.

In [ ]:
def standardize_nationality(column):
    """
    Takes the DataFrame and standardizes UK values
    """
    uk_variants = ["United Kingdom","United Kingdom of Great Britain and Ireland","United Kingdom of Great Britain","British Empire"]
    
    for variant in uk_variants:
        if isinstance(column, str):  #error handling for null values
            if variant in column:
                return "United Kingdom"

    return column #keeps old values in case of no match

df["country of citizenship"] = df["country of citizenship"].apply(standardize_nationality)
df.head(10)

#### Now we make two DataFrames with a selection of novels authored only by UK and US authors

In [ ]:
uk_df = df[df["country of citizenship"] == "United Kingdom"]
us_df = df[df["country of citizenship"] == "United States"]

us_df.head(3)

In [ ]:
uk_df.head(3)

In [ ]:
def categorize_genre(genre_column):
    """
    Takes a genre string and returns a standardized genre:
    'science fiction', 'fantasy', 'both', or '' if neither.
    """
    # Ensure the genre string is lowercase for consistent matching
    genre_lower = str(genre_column).lower()
    
    has_scifi = "science fiction" in genre_lower #boolean
    has_fantasy = "fantasy" in genre_lower #boolean

    if has_scifi and has_fantasy:
        return "fantasy, science fiction"
    elif has_scifi:
        return "science fiction"
    elif has_fantasy:
        return "fantasy"
    else:
        return ""  # empty string if neither

In [ ]:
stand_genres = df['genre'].apply(categorize_genre)

df.insert(2, "standardized genre", stand_genres)

In [ ]:
df.head(10)

In [ ]:
#this codes changes the datatype of the publication date column to datetime
df['publication date'] = pd.to_datetime(df['publication date'], errors='coerce', utc=True)
df['publication date'].dtype

In [ ]:
#Create a seelction of 19th century novels
mask = (df['publication date'] >= '1801-01-01') & (df['publication date'] < '1901-01-01')
df_19c = df[mask]
df_19c.head(10)